# Materialize audit JSONL as a Delta table

Reads the dated JSON Lines exports written by `audit_events_splunk.ipynb` and materializes them as the separate Delta table `audit_events_splunk_jsonl`. Attach the Lakehouse containing `Files/splunk_export` as this notebook's default Lakehouse before running. The source folder is read-only; the existing `audit_events_splunk` table is not modified.

In [ ]:
# The exporter writes one Spark part file under each date folder:
# Files/splunk_export/YYYY/MM/DD/audit_events.jsonl/part-*.json
jsonl_source = "Files/splunk_export/*/*/*/audit_events.jsonl/part-*.json"
jsonl_table = "audit_events_splunk_jsonl"

jsonl_df = spark.read.json(jsonl_source)
if not jsonl_df.columns:
    raise ValueError(f"No columns found in JSONL export at {jsonl_source}")

source_row_count = jsonl_df.count()
if source_row_count == 0:
    raise ValueError(f"No JSONL records found in {jsonl_source}")

(
    jsonl_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(jsonl_table)
)

table_df = spark.read.table(jsonl_table)
table_row_count = table_df.count()
if source_row_count != table_row_count:
    raise RuntimeError(
        f"Row-count mismatch: source={source_row_count}, table={table_row_count}"
    )

print(f"Materialized {table_row_count} rows in {jsonl_table}")
table_df.printSchema()
display(table_df.limit(10))